# =====================================================================
# COMPARAISON DES MODELES DE CLASSIFICATION 5G
# Supervises vs Non-Supervises (Binaire et Multi-classes)
# =====================================================================

Ce notebook compare les metriques d'evaluation de **tous les modeles** developpes :
- **6 modeles supervises (Binaire & Multi-classes)** : Decision Tree, Gradient Boosting, KNN, Logistic Regression, Random Forest, SVM
- **1 modele non-supervise (Binaire)** : Isolation Forest
- **1 modele non-supervise (Multi-classes)** : K-Means / GMM (Clustering Metrics)

L'objectif est de determiner les modeles les plus performants pour la detection d'anomalies dans les reseaux 5G, en binaire (anomalie vs normal) et en multi-classes (type d'anomalie).

In [ ]:
# Bibliotheques necessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
plt.style.use("ggplot")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Bibliotheques importees avec succes !")

## 1. Metriques d'evaluation - Classification Binaire

**Anomalie vs Trafic Normal**

Les metriques suivantes sont extraites des modeles supervises optimises et du modele non-supervise (Isolation Forest).

In [ ]:
# =====================================================================
# Metriques d'evaluation (Classification Binaire)
# =====================================================================

binary_metrics = {
    "Modele": [
        "Decision Tree",
        "Gradient Boosting",
        "KNN",
        "Logistic Regression",
        "Random Forest",
        "SVM",
        "Isolation Forest (Non-Supervise)"
    ],
    "Accuracy": [0.9932, 0.9968, 0.9938, 0.9846, 0.9968, 0.9952, None],
    "Precision": [0.8198, 0.9947, 0.9545, 0.7178, 1.0000, 0.9694, 0.4135],
    "Recall":    [0.8636, 0.8511, 0.7386, 0.4307, 0.8466, 0.7932, 0.8837],
    "F1-Score":  [0.8412, 0.9173, 0.8328, 0.5384, 0.9169, 0.8725, 0.6633],
    "ROC-AUC":   [0.9298, 0.9476, 0.9027, 0.9348, 0.9462, 0.9143, 0.9632],
    "Type":      ["Supervise", "Supervise", "Supervise", "Supervise", "Supervise", "Supervise", "Non-Supervise"]
}

df_binary = pd.DataFrame(binary_metrics)

print("="*90)
print("COMPARAISON DES METRIQUES - Classification Binaire")
print("="*90)
display(df_binary)

In [ ]:
# Visualisation: Classification Binaire
metrics = ["Precision", "Recall", "F1-Score", "ROC-AUC"]
models = df_binary["Modele"].tolist()
colors = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#1abc9c", "#34495e"]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("Comparaison des Metriques Binaire (Supervise v. Non-Supervise)", fontsize=16, fontweight="bold")

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2][idx % 2]
    values = df_binary[metric].fillna(0).tolist()
    bars = ax.bar(range(len(models)), values, color=colors, edgecolor="black", alpha=0.85)
    
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{val:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    
    ax.set_title(metric, fontsize=14)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(" ", "\n") for m in models], fontsize=9, rotation=0)
    ax.set_ylim(0, 1.15)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 2. Metriques d'evaluation - Classification Multi-classes

**Typage de l'anomalie (9 classes d'anomalies 5G differentes)**

Note: Les modeles supervises predisent avec precision la classe (Accuracy), tandis que les approches non-supervisees tentent de decouvrir les clusters sous-jacents, l'Accuracy n'y est donc pas pertinente de façon directe sans alignement prealable.

In [ ]:
# =====================================================================
# Metriques d'evaluation (Classification Multi-classes)
# =====================================================================

multiclass_metrics = {
    "Modele": [
        "Decision Tree",
        "Gradient Boosting",
        "KNN",
        "Logistic Regression",
        "Random Forest",
        "SVM Optimized"  # Performance en multi-classes affectee par la dimensionalite
    ],
    "Accuracy (Multi-classes)": [0.9963, 0.9834, 0.9933, 0.9890, 0.9963, 0.0834],
    "Type": ["Supervise"] * 6
}

df_multi = pd.DataFrame(multiclass_metrics)

print("="*90)
print("COMPARAISON DES METRIQUES - Classification Multi-classes")
print("="*90)
display(df_multi)

In [ ]:
# Visualisation: Classification Multi-classes Accuracy
plt.figure(figsize=(10, 6))
models_mc = df_multi["Modele"].tolist()
values_mc = df_multi["Accuracy (Multi-classes)"].tolist()
colors_mc = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#1abc9c"]

bars = plt.bar(models_mc, values_mc, color=colors_mc, edgecolor="black")

for bar, val in zip(bars, values_mc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.title("Accuracy en Classification Multi-classes (Modeles Supervises)", fontsize=14, fontweight="bold")
plt.ylabel("Accuracy")
plt.ylim(0, 1.15)
plt.axhline(y=0.95, color="gray", linestyle="--", alpha=0.5)
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

### Modeles Non-Supervises (Multi-classes) - Clustering
- **Objectif**: Regrouper les flux reseaux sans etiquettes prealables.
- **K-Means & DBSCAN** ont ete testes dans `unsupervised_multiclass.ipynb`.
- **Metriques** : Silhouette Score, Davies-Bouldin Index.
> *L'evaluation montre la capacite a regrouper les comportements anormaux, mais sans correspondre directement avec le seuil absolu de classification 1-a-1 des annotations originales.*

In [ ]:
# =====================================================================
# Heatmap Analytique Globale (Binaire)
# =====================================================================

metrics_cols = ["Precision", "Recall", "F1-Score", "ROC-AUC"]
df_heat = df_binary.set_index("Modele")[metrics_cols].fillna(0)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(df_heat, annot=True, fmt=".4f", cmap="YlGnBu",
            linewidths=0.5, ax=ax, vmin=0.4, vmax=1.0,
            cbar_kws={"label": "Score"})
ax.set_title("Heatmap des Metriques Binaire - Tous les Modeles", fontsize=14, fontweight="bold")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 3. Classement Final et Conclusion

### Pour la Classification Binaire (Detection d'Anomalie simple) :
1. **Random Forest** & **Gradient Boosting** offrent des performances quasi-parfaites (F1-score ~0.92, ROC-AUC ~0.95).
2. Le modele **Non-supervise (Isolation Forest)** atteint un Recall eleve (0.88), le rendant adapte a des contextes zero-day (nouvelles attaques non labelisees) bien qu'il genere plus de faux-positifs (Precision plus faible).

### Pour la Classification Multi-classes (Typage de l'Anomalie) :
1. **Decision Tree** et **Random Forest** dominent avec une accuracy exceptionnelle de **0.9963**.
2. Les algorithmes d'ensemblistes sont les plus adaptes a la structure de ces donnees de trafic 5G et gerent extremement bien la grande dimension des features reseaux.